In [1]:
import re
import regex
import requests
from collections import defaultdict, deque
import json
import numpy as np
import pandas as pd
from PyPDF2 import PdfReader
import camelot
import os
import requests
import io
from collections import OrderedDict
from tabulate import tabulate

In [4]:
#staging algorithm at https://github.com/imsweb/staging-client-java/tree/master?tab=readme-ov-file
algorithm = "eod_public-3.3"
algorithm_version = '3.3'

#download algorithm files if needed
if not os.path.exists(algorithm):
    !wget https://github.com/imsweb/staging-client-java/releases/download/v11.7.1/eod_public-3.3.zip
    !unzip eod_public-3.3.zip -d eod_public-3.3

schemas_path = os.path.join(algorithm, "schemas")
tables_path = os.path.join(algorithm, "tables")

def print_dict(json_obj):
    print(json.dumps(json_obj, indent=2))

#create a json file with id and notes mapping for quick retrieval
#this is used to get the primary site name from code later on
grade_clin_manuals = {}
grade_path_manuals = {}
for file in os.listdir(schemas_path):
    if ".json" in file:
        with open(os.path.join(schemas_path,file), "r") as f:
            data = json.load(f)
            primary_site = data["id"]
            grade_clin_manuals[primary_site] = {}
            grade_path_manuals[primary_site] = {}
            for item in data["inputs"]:
                if item.get("key") == "grade_clin":
                    grade_clin_table_name = item["table"]
                if item.get("key") == "grade_path":
                    grade_path_table_name = item["table"]
            with open(os.path.join(tables_path, grade_clin_table_name+".json"), "r") as f:
                table = json.load(f)
                grade_clin_manuals[primary_site]["notes"] = table["notes"]
                grade_clin_manuals[primary_site]["definition"] = table["definition"]
                grade_clin_manuals[primary_site]["rows"] = table["rows"]
            with open(os.path.join(tables_path, grade_path_table_name+".json"), "r") as f:
                table = json.load(f)
                grade_path_manuals[primary_site]["notes"] = table["notes"]
                grade_path_manuals[primary_site]["definition"] = table["definition"]
                grade_path_manuals[primary_site]["rows"] = table["rows"]
            
with open(f"dict_grade_clin.json", "w") as f:
    json.dump(grade_clin_manuals, f, indent=2)
with open(f"dict_grade_path.json", "w") as f:
    json.dump(grade_path_manuals, f, indent=2)

In [5]:
reader = PdfReader("SPCSM_2024_MainDoc.pdf")

staging_manual_dict = defaultdict(list)
for page in reader.pages:
    text = page.extract_text()
    if "NAACCR Item #:" in text:
        lines = np.array(text.split("\n"))

In [6]:
text = ""
for page_num in range(2, 8):
    text += reader.pages[page_num].extract_text() + "\n"

header_footer = r"SEER Program Coding and Staging Manual 2024\s+September 2023\s+Table of Contents  \d+\s+"
text = re.sub(header_footer, "", text)
text = re.sub(r"Table of Contents\s+", "", text)
text = text.replace("- ", "-").replace(" -", "-")

#print(text)

In [7]:
multi_split = deque(re.split(r"(\d+\s+$)", text, flags=re.MULTILINE))

toc = []
while multi_split:
    key = multi_split.popleft()
    if key == "":
        continue

    val = multi_split.popleft()
    toc.append((key.strip(), int(val.strip())))

#toc

In [8]:
naaccr_pages = []
for page in range(len(reader.pages)):
    page_text = reader.pages[page].extract_text()
    item_match = re.search(r"NAACCR Item #: .+", page_text)
    if item_match is not None:
        item_id = item_match.group().replace("NAACCR Item #:", "").strip().replace(" ", "")
        naaccr_obj = {"naaccr_id": item_id, "start_page": page}

        if page == len(reader.pages) - 1:
            naaccr_obj.update({"end_page": len(reader.pages)})
        
        if naaccr_pages:
            naaccr_pages[-1].update({"end_page": page})
        
        naaccr_pages.append(naaccr_obj)

naaccr_pages = {obj.pop("naaccr_id"): obj for obj in naaccr_pages}

In [9]:
#390: Dat of diagnosis
#400: primary site - refer to STM for site specific
#410: laterality
#440: grade x
#3843: grade clinical - from schema - captured above
#3844: grade pathological - from schema - captured above
#522: histology - refer to STM for site specific
#759: summary stage 2000 for < 2018  x
#764: summary stage 2018
#523: behavior

reqd_ids = [390, 400, 410, 764, 523]

def clean_section_text(section_text):
    header = re.compile(r"^SEER Program Coding and Staging Manual 2024.*", re.MULTILINE)
    footer = re.compile(r"^September 2023.*", re.MULTILINE)
    section_text = re.sub(header, "", section_text)
    section_text = re.sub(footer, "", section_text)
    section_text = section_text.replace("- ", "-").replace(" -", "-").strip()
    
    return section_text


naaccr_coding = {}
for naaccr_id, page_nums in naaccr_pages.items():
    try:
        item_dict = {}
        
        section_text = "\n".join(reader.pages[page].extract_text() for page in range(page_nums["start_page"], page_nums["end_page"]))
        section_text = clean_section_text(section_text)

        if "," in naaccr_id:
            naaccr_ids = naaccr_id.replace(" ", "").split(",")
            n_ids = len(naaccr_ids)
            #print("NAACCR Ids: ", naaccr_ids, " pages: ", page_nums)
            name_match = regex.search(r"^NAACCR Name:\s*(.+)", section_text, re.MULTILINE)
            xml_match = regex.search(r"^XML NAACCR ID:\s*(.+)", section_text, re.MULTILINE)

            if "," in xml_match.group(1):
                naaccr_names = name_match.group(1).replace(" ","").split(",")
                naaccr_xmls = xml_match.group(1).replace(" ","").split(",")
                description = section_text.split(xml_match.group())[-1].strip()
            else:       
                name_match = regex.search(r"^NAACCR Name:\s*([^\n]+(\n[^\n]+)*)$", section_text, re.MULTILINE).group(1)
                xml_match = regex.search(r"^XML NAACCR ID:\s*([^\n]+(\n[^\n]+)*)$", section_text, re.MULTILINE).group(1)
                naaccr_names = name_match.split("\n")[:n_ids]
                naaccr_xmls = xml_match.split("\n")[:n_ids]
                description = section_text.split(naaccr_xmls[-1])[-1].strip()
                naaccr_names = [x.strip() for x in naaccr_names]
                naaccr_xmls = [x.strip() for x in naaccr_xmls]
                
                #print("##########name_match", naaccr_names)
                #print("##########xml_match", naaccr_xmls)
                #print("##########description: ", item_dict["description"])            
            for n_id, n_name, n_xml in zip(naaccr_ids, naaccr_names, naaccr_xmls):
                if int(n_id) in reqd_ids:
                    naaccr_coding[int(n_id)] = {}
                    naaccr_coding[int(n_id)]["naaccr_name"] = n_name
                    naaccr_coding[int(n_id)]["naaccr_xml"] = n_xml
                    naaccr_coding[int(n_id)]["description"] = description
        else:
            # id_match = regex.search(r"^(NAACCR Item #: ){e<=2}(?P<naaccr_id>\S+)", section_text, re.MULTILINE)
            name_match = regex.search(r"^(NAACCR Name: ){e<=2}(?P<naaccr_name>.+)", section_text, re.MULTILINE)
            xml_match = regex.search(r"^(XML NAACCR ID: ){e<=2}(?P<naaccr_xml>.+)", section_text, re.MULTILINE)
            
            item_dict.update(name_match.groupdict())
            item_dict.update(xml_match.groupdict())
            item_dict["description"] = section_text.split(xml_match.group())[-1].strip()
            item_dict["naaccr_xml"] = item_dict["naaccr_xml"].replace(" ", "")
            if int(naaccr_id) in reqd_ids:
                naaccr_coding[int(naaccr_id)] = item_dict
            
    except:
        print("exception: ", naaccr_id)

with open("seer_manual_selected_codes.json", "w") as f:
    json.dump(naaccr_coding, f, indent=2)

In [12]:
if not os.path.exists("STM_Combined.pdf"):
    !wget https://seer.cancer.gov/tools/solidtumor/current/STM_Combined.pdf
    
def get_STR_TOC_Adv()->dict:
    # Parse https://seer.cancer.gov/tools/solidtumor/current/STM_Combined.pdf
    reader = PdfReader("STM_Combined.pdf")

    toc = OrderedDict()
    #print(reader.outline)
    page_nums = []
    prev_section, prev_sub_section = None, None
    for o in reader.outline[1]:
        if not isinstance(o, list):
            section_name = o.title.strip()
            toc[section_name] = {}
            if prev_section and prev_sub_section:
                toc[prev_section][prev_sub_section]['end_page'] = reader.get_destination_page_number(o)-1
            prev_section = section_name
        else:
            prev_sub_section = None
            for l in o:
                if not isinstance(l, list):
                    sub_section_name = l.title.strip()
                    toc[section_name][sub_section_name] = {}
                    toc[section_name][sub_section_name]['start_page'] = reader.get_destination_page_number(l)
                    if prev_sub_section:
                        toc[section_name][prev_sub_section]['end_page'] = reader.get_destination_page_number(l)-1
                prev_sub_section = sub_section_name                        
    return toc
str_toc = get_STR_TOC_Adv()
#print(str_toc)

--2026-03-19 14:23:51--  https://seer.cancer.gov/tools/solidtumor/current/STM_Combined.pdf
Resolving seer.cancer.gov (seer.cancer.gov)... 131.226.202.20, 2604:7ac0:101::17
Connecting to seer.cancer.gov (seer.cancer.gov)|131.226.202.20|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9162146 (8.7M) [application/pdf]
Saving to: ‘STM_Combined.pdf’

STM_Combined.pdf    100%[===================>]   8.74M  17.2MB/s    in 0.5s    

2026-03-19 14:23:52 (17.2 MB/s) - ‘STM_Combined.pdf’ saved [9162146/9162146]



In [19]:
def get_STR_Table(tissue_type: str, toc: dict, table_type: str) -> pd.DataFrame:
    subsections = list(toc[tissue_type].keys())
    start_page = toc[tissue_type][subsections[0]]['start_page']
    end_page = toc[tissue_type][subsections[-1]]['end_page']
        
    pages = f"{start_page}-{end_page}"
    print(pages)
    tables = camelot.read_pdf("STM_Combined.pdf", pages=pages, flavor='lattice', flag_size=True)
    tables = [t for t in tables if not t.df.empty]
    pd_tables = []
    for t in tables:
        t_df = t.df
        t_df.columns = t_df.iloc[0].tolist()
        t_df[t_df.columns[-1]] = t_df[t_df.columns[-1]].apply(lambda x: re.sub(r"<s>.*?</s>", "", str(x)))
        t_df = t_df.iloc[1:]
        pd_tables.append(t_df)
    print("Found ",len(tables), " tables")

    merged_tables = [pd_tables[0]]
    for i in range(len(pd_tables)-1):
        if merged_tables[-1].columns.equals(pd_tables[i+1].columns):
            merged_tables[-1] = pd.concat([merged_tables[-1], pd_tables[i+1]], axis=0, ignore_index=True)
        else:
            merged_tables.append(pd_tables[i+1])
    return merged_tables[table_no]

In [ ]:
primary_sites = ["Urinary", "Breast", "Colon", "Lung"]
table_nos = [0, 0, 2, 0]
for site, table_no in zip(primary_sites, table_nos):
    print(f"Extracting table for site {site}")
    df = get_STR_Table(site, str_toc, table_no)
    #print(tabulate(df, headers = 'keys', tablefmt = 'grid'))
    with open(f"{site}_sitegroupinstr.json", "w") as f:
        json.dump(df.to_json(), f, indent=2)